# Inline Mesh Viewer

`plot_mesh_threejs` — interactive Three.js `.msh` viewer, zero Python dependencies, runs entirely in the browser.

In [ ]:
import gdsfactory as gf
from ihp import LAYER, PDK

PDK.activate()


@gf.cell
def gsg_electrode(
    length: float = 800,
    s_width: float = 20,
    g_width: float = 40,
    gap_width: float = 15,
    layer=LAYER.TopMetal2drawing,
) -> gf.Component:
    c = gf.Component()
    r1 = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
    r1.move((0, (g_width + s_width) / 2 + gap_width))
    _r2 = c << gf.c.rectangle((length, s_width), centered=True, layer=layer)
    r3 = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
    r3.move((0, -(g_width + s_width) / 2 - gap_width))
    c.add_port(
        name="o1",
        center=(-length / 2, 0),
        width=s_width,
        orientation=0,
        port_type="electrical",
        layer=layer,
    )
    c.add_port(
        name="o2",
        center=(length / 2, 0),
        width=s_width,
        orientation=180,
        port_type="electrical",
        layer=layer,
    )
    return c


c = gsg_electrode()
c

In [ ]:
from gsim.palace import DrivenSim

sim = DrivenSim()
sim.set_output_dir("./mesh-viewer-demo")
sim.set_geometry(c)
sim.set_stack(substrate_thickness=2.0, air_above=300.0)
sim.add_cpw_port("o1", layer="topmetal2", s_width=20, gap_width=15)
sim.add_cpw_port("o2", layer="topmetal2", s_width=20, gap_width=15)
sim.set_driven(fmin=1e9, fmax=100e9, num_points=300)
sim.mesh(preset="default", planar_conductors=False)

In [ ]:
from gsim.viz import plot_mesh_threejs

plot_mesh_threejs("mesh-viewer-demo/palace.msh")